# 🏠 House Price Prediction Project
> **Objective**: Building a Linear Regression model to estimate property values using the USA Housing dataset.
---
> **78% accurate**
---

## 📥 1. Data Loading & Initial Inspection
In this section, we import the necessary libraries and load the raw dataset to understand its structure, data types, and potential issues.

In [551]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error


In [552]:
df=pd.read_csv("data.csv")

In [553]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4600 entries, 0 to 4599
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   date           4600 non-null   object 
 1   price          4600 non-null   float64
 2   bedrooms       4600 non-null   float64
 3   bathrooms      4600 non-null   float64
 4   sqft_living    4600 non-null   int64  
 5   sqft_lot       4600 non-null   int64  
 6   floors         4600 non-null   float64
 7   waterfront     4600 non-null   int64  
 8   view           4600 non-null   int64  
 9   condition      4600 non-null   int64  
 10  sqft_above     4600 non-null   int64  
 11  sqft_basement  4600 non-null   int64  
 12  yr_built       4600 non-null   int64  
 13  yr_renovated   4600 non-null   int64  
 14  street         4600 non-null   object 
 15  city           4600 non-null   object 
 16  statezip       4600 non-null   object 
 17  country        4600 non-null   object 
dtypes: float

## 🛠️ 2. Data Cleaning & Feature Engineering
To ensure the model is robust, we perform the following refinements:
* **Filtering**: Removing zero-priced entries and dropping non-essential columns (`street`, `country`).
* **Feature Extraction**: 
    * Extracting `zip` codes from the `statezip` column.
    * Converting `yr_renovated` into a binary `is_renovated` feature.
* **Redundancy Removal**: Dropping columns like `sqft_above` and `sqft_basement` to minimize multicollinearity.
* **Outlier Mitigation**: Filtering out the top 1% of extreme prices to prevent the regression line from being skewed by "luxury" anomalies.

In [554]:
df = df[df['price'] > 0].drop(columns=['street', 'country'])

df[['state', 'zip']] = df['statezip'].str.split(r'\s+', expand=True)
df = df.drop(columns=['statezip', 'state'])

df['is_renovated'] = (df['yr_renovated'] > 0).astype(int)
df = df.drop(columns='yr_renovated')

df = df.drop(columns=['sqft_above', 'sqft_basement', 'date', 'city'])

limit = df["price"].quantile(0.99)
df = df[df["price"] < limit]


In [555]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4505 entries, 0 to 4599
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   price         4505 non-null   float64
 1   bedrooms      4505 non-null   float64
 2   bathrooms     4505 non-null   float64
 3   sqft_living   4505 non-null   int64  
 4   sqft_lot      4505 non-null   int64  
 5   floors        4505 non-null   float64
 6   waterfront    4505 non-null   int64  
 7   view          4505 non-null   int64  
 8   condition     4505 non-null   int64  
 9   yr_built      4505 non-null   int64  
 10  zip           4505 non-null   object 
 11  is_renovated  4505 non-null   int64  
dtypes: float64(4), int64(7), object(1)
memory usage: 457.5+ KB


## 📊 3. Exploratory Data Analysis (EDA)
Visualizing the relationships between features and the target variable (`price`):
* **Scatter Plots**: Analyzing the linearity and distribution of numerical features.
* **Box Plots**: Inspecting the variance of price across categorical dimensions (geographic location).

In [ ]:
import matplotlib.pyplot as plt

num_cols = df.select_dtypes(include='number').columns
num_cols = num_cols.drop('price')

for col in num_cols:
  plt.figure(figsize=(6,4))
  plt.scatter(df[col], df['price'], s=5, alpha=0.2)
  plt.ylim(0, df['price'].quantile(0.99))
  plt.xlabel(col)
  plt.ylabel('price')
  plt.grid(alpha=0.3)
  plt.show()

In [ ]:
cat_cols = df.select_dtypes(include='object').columns

for col in cat_cols:
  plt.figure(figsize=(8,4))
  df.boxplot(column='price', by=col)
  plt.ylim(0, df['price'].quantile(0.99))
  plt.title(f'price vs {col}')
  plt.suptitle('')
  plt.show()

## ⚙️ 4. Model Preparation & Implementation
This stage involves transforming the data and executing the machine learning pipeline:

* **Categorical Encoding**: Converting `zip` codes into dummy variables via One-Hot Encoding.
* **Target Scaling**: Applying `np.log1p` to the price to normalize distribution and handle skewness.
* **Data Splitting**: Partitioning the dataset into **80% Training** and **20% Testing** sets.
* **Feature Scaling**: Implementing `StandardScaler` on numerical inputs to ensure all features contribute equally to the model coefficients.
* **Model Fitting**: Training the Linear Regression model on the processed training data.

In [558]:
df = pd.get_dummies(df, columns=['zip'], drop_first=True)
df['price'] = np.log1p(df['price'])

In [559]:
X = df.drop('price', axis=1)
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

num_cols = [
    'bedrooms','bathrooms','sqft_living','sqft_lot',
    'floors','waterfront','view','condition','yr_built','is_renovated'
]

scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("R2 =", r2_score(y_test, y_pred))
print("RMSE =", np.sqrt(mean_squared_error(y_test, y_pred)))

R2 = 0.7814052775769563
RMSE = 0.23011816632255


In [560]:
y_pred_ori = np.expm1(y_pred)
y_test_ori = np.expm1(y_test)

r2_log = r2_score(y_test, y_pred)
r2_ori = r2_score(y_test_ori, y_pred_ori)

rmse_log = np.sqrt(mean_squared_error(y_test, y_pred))
rmse_ori = np.sqrt(mean_squared_error(y_test_ori, y_pred_ori))

print(f"""
      R2 log = {r2_log}
      R2 ori = {r2_ori}
      RMSE log = {rmse_log}
      RMSE ori = {rmse_ori}
      """)


      R2 log = 0.7814052775769563
      R2 ori = 0.7718602546029203
      RMSE log = 0.23011816632255
      RMSE ori = 133315.51032772465
      


## 📝 5. Conclusion & Final Insights

### 📊 Performance Metrics
| Metric | Log Scale (Training) | Original Scale (USD) |
| :--- | :---: | :---: |
| **R-Squared (R²)** | **0.7814** | **0.7719** |
| **RMSE** | **0.2301** | **$133,315.51** |

---

### 🔍 Key Takeaways

1. **Successful Normalization**
   Applying **Log Transformation** and **Outlier Filtering** was instrumental in model stability. Removing the top 1% of outliers alone drastically improved the **RMSE**, bringing it down to a realistic **$133k**.

2. **Geographic Significance**
   The model achieved a strong **R² of 0.78**, confirming that **Zip Code** is a primary driver of property value. This highlights that location remains the most significant predictor in this dataset.

3. **Optimized Feature Set**
   By binarizing the renovation data and removing redundant square footage metrics, we maintained high predictive power with a cleaner, more efficient feature set.

---
**Next Steps**: 
* Experiment with **Polynomial Features** to capture non-linear relationships.
* Test **Tree-based models** (Random Forest/XGBoost) for potentially higher accuracy.